In [1]:
import torch
from argparse import Namespace
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from pandas import Series

torch.serialization.add_safe_globals([
    Namespace,
    Phonemer_Tokenizer_Recombination,
    Series,
])


/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._r

In [2]:
import torch

# Monkey-patch torch.load to default to weights_only=False (Torch 2.6+ defaults to True)
_orig_torch_load = torch.load

def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load_no_weights_only
print("✅ Patched torch.load to default weights_only=False for this kernel session.")


✅ Patched torch.load to default weights_only=False for this kernel session.


In [3]:
import wandb, random

run = wandb.init(
    entity="krishrawat0222-f",  # 👈 replace with your wandb username or team name
    project="DeepfakeDetectionRenewed",
    config={
        "test_run": True,
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "dummy",
        "epochs": 10,
    },
)

for epoch in range(10):
    acc = 1 - 2**-epoch - random.random() / epoch if epoch > 0 else 0.1
    loss = 2**-epoch + random.random() / (epoch + 1)
    run.log({"acc": acc, "loss": loss})

run.finish()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: yashaspatil (krishrawat0222-f) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


acc,▂▁▅▇▇█████
loss,█▃▃▂▁▁▁▁▁▁
acc,0.91596
loss,0.08651


## Phoneme Recognition Model

In [4]:
import os, sys
# Add the directory containing the notebook to sys.path
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

In [5]:
from phoneme_GAT.phoneme_model import BaseModule, load_phoneme_model, optim_param

/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [6]:
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination

1. You can download the pretrained phoneme recognition model in [google drive](https://drive.google.com/file/d/1SbqynkUQxxlhazklZz9OgcVK7Fl2aT-z/view?usp=drive_link).
2. Change `pretrained_path` to you own custom path.
3. Remember to change the `pretrained_path` and `vocab_path` in the `load_phoneme_model` function of `phoneme_GAT.phoneme_model`.

In [7]:
network_param = Namespace(
    network_name="WavLM",
    pretrained_path="pretrained/best-epoch=42-val-per=0.407000.ckpt",
    pretrained_name="microsoft/wavlm-base",   # ✅ add this
    freeze=True,
    freeze_transformer=True,
    eos_token="</s>",
    bos_token="<s>",
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|",
    vocab_size=200,
)


To build the phoneme recognition model,
1. you must specify the pretrained_path!!!! Please download the provided pretrained phoneme model; or you can train yourself model through `train_phoneme_model.py`.
2. in the `load_phoneme_model` function, you have to change the correct `vocab_path`

In [8]:
from phoneme_GAT.phoneme_model import BaseModule, load_phoneme_model, optim_param

total_num_phonemes = 687  ## 198, or 687

phoneme_model = load_phoneme_model(
    network_name=network_param.network_name,
    pretrained_path=network_param.pretrained_path,
    total_num_phonemes=total_num_phonemes,
)
assert len(phoneme_model.tokenizer.total_phonemes) == total_num_phonemes

Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['lm_head.bias', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.weight', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


# Load model

## Audio model

In [9]:
from phoneme_GAT.modules import Phoneme_GAT_lit,Phoneme_GAT

In [10]:
audio_model = Phoneme_GAT(
    backbone='wavlm',
    use_raw=0,
    use_GAT=1,
    n_edges=10,
)

Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['lm_head.bias', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.weight', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


Generate a random audio to test the model.

In [11]:
import os

# ----------------------------
# Load HF token from secret.txt
# ----------------------------
def load_hf_token(path="secret.txt"):
    if os.path.exists(path):
        with open(path, "r") as f:
            return f.read().strip()
    return None

DATASET_NAME   = "Bisher/ASVspoof_2019_LA"
CACHE_DIR      = "./data/asvspoof_2019_la"
SUBSET_SAMPLES = None

HF_TOKEN = load_hf_token()  # 🔥 auto-loads from secret.txt

print("Dataset name    :", DATASET_NAME)
print("Cache dir       :", CACHE_DIR)
print("Subset samples  :", SUBSET_SAMPLES)
print("HF token present:", "✅ Yes" if HF_TOKEN else "❌ No (use secret.txt or HF login)")


Dataset name    : Bisher/ASVspoof_2019_LA
Cache dir       : ./data/asvspoof_2019_la
Subset samples  : None
HF token present: ✅ Yes


In [12]:
from datasets import load_dataset, Audio
from collections import Counter

print("=" * 80)
print("Downloading ASVspoof 2019 LA to cache (once) and verifying decoding…")
print("=" * 80)

# Use HF TEST split as the pool we'll train from (since it's bigger)
hf_train_pool = load_dataset(
    DATASET_NAME,          # e.g. "LanceaKing/asvspoof2019"
    split="test",
    cache_dir=CACHE_DIR,
    token=HF_TOKEN if HF_TOKEN else None,
)

print("Columns in HF split:", hf_train_pool.column_names)
print("Features:", hf_train_pool.features)

# Ensure audio comes out at 16 kHz (consistent with model)
hf_train_pool = hf_train_pool.cast_column("audio", Audio(sampling_rate=16000))

# Handle the case where SUBSET_SAMPLES can be None
if SUBSET_SAMPLES is None:
    subset_size = len(hf_train_pool)
else:
    subset_size = min(SUBSET_SAMPLES, len(hf_train_pool))

hf_train_small = hf_train_pool.shuffle(seed=42).select(range(subset_size))

print(f"✓ Cached under: {CACHE_DIR}")
print(f"✓ HF TEST split size (train pool): {len(hf_train_pool)} | (subset): {len(hf_train_small)}")

# Inspect first sample keys & audio metadata
s0 = hf_train_small[0]
print("\nFirst sample keys:", list(s0.keys()))
a0 = s0["audio"]
print("Audio fields present:", list(a0.keys()))
print("Audio sampling_rate:", a0.get("sampling_rate"))
print(
    "Audio array dtype/len:",
    type(a0.get("array")).__name__,
    len(a0.get("array")) if a0.get("array") is not None else None,
)

# ----- Label / key inspection (robust) -----

# Decide which column we should treat as the label-ish thing
label_col = None
for cand in ["label", "key", "class", "target"]:
    if cand in hf_train_small.column_names:
        label_col = cand
        break

if label_col is None:
    print("\n[WARN] No obvious label column found; available columns:", hf_train_small.column_names)
else:
    print(f"\nUsing '{label_col}' as label-like column for sanity check.")
    raw_vals = [ex[label_col] for ex in hf_train_small]
    print(f"Raw '{label_col}' value counts:", Counter(raw_vals))

    # Try to map to numeric 0/1 for convenience, but handle both strings and ints
    key_to_int = {"bonafide": 0, "spoof": 1, "real": 0, "fake": 1}

    numeric_labels = []
    for v in raw_vals:
        if isinstance(v, int):
            numeric_labels.append(v)
        elif isinstance(v, str):
            if v in key_to_int:
                numeric_labels.append(key_to_int[v])
            else:
                # fallback: hash to 0/1 just so Counter works
                numeric_labels.append(hash(v) % 2)
        else:
            # weird type, just cast to int if possible
            try:
                numeric_labels.append(int(v))
            except Exception:
                numeric_labels.append(0)

    print("Numeric label counts (best-effort 0/1 mapping):", Counter(numeric_labels))

print("=" * 80)


Columns in HF split: ['speaker_id', 'audio_file_name', 'audio', 'system_id', 'key']
Features: {'speaker_id': Value(dtype='string', id=None), 'audio_file_name': Value(dtype='string', id=None), 'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'system_id': Value(dtype='string', id=None), 'key': ClassLabel(names=['bonafide', 'spoof'], id=None)}
✓ Cached under: ./data/asvspoof_2019_la
✓ HF TEST split size (train pool): 71237 | (subset): 71237

First sample keys: ['speaker_id', 'audio_file_name', 'audio', 'system_id', 'key']
Audio fields present: ['path', 'array', 'sampling_rate']
Audio sampling_rate: 16000
Audio array dtype/len: ndarray 26536

Using 'key' as label-like column for sanity check.
Raw 'key' value counts: Counter({1: 63882, 0: 7355})
Numeric label counts (best-effort 0/1 mapping): Counter({1: 63882, 0: 7355})


In [13]:
from loader import get_dataloader, get_train_dataloader, get_eval_dataloader

val_dataloader = get_eval_dataloader(
    source="hf",
    split="validation",
    hf_name=DATASET_NAME,
    batch_size=20,
    hf_cache_dir=CACHE_DIR,
    limit=3000,
    hf_token=HF_TOKEN if HF_TOKEN else None,
)

/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/loader.py:18: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


✓ HF Bisher/ASVspoof_2019_LA:validation [eval] → 3000 samples (balance=True)


In [ ]:
!pip install encodec --quiet

In [34]:
val_dataloader = DataLoader(
    val_dataloader.dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [79]:
# ─────────────────────────────────────────────────────────────────────────────
# AUGMENTATION ATTACK SUITE  ·  Run ONE augmentation at a time + audition 10 clips
# ─────────────────────────────────────────────────────────────────────────────
import random, torch, numpy as np, math
import IPython.display as ipd
from IPython.display import display
import torchaudio
import torchaudio.functional as F
from fractions import Fraction

SAMPLE_RATE = 16_000
LISTEN_N    = 10
SEED        = 42
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

# ── ✏️  CHANGE THESE ─────────────────────────────────────────────────────────
ACTIVE_AUG = "pitch_down"
# Options:
#   "pitch_up"        – pitch shift up N semitones
#   "pitch_down"      – pitch shift down N semitones
#   "phase_noise"     – small random phase perturbation
#   "amp_scale"       – fixed amplitude multiplier
#   "additive_noise"  – white noise at target SNR (dB)
#   "time_stretch"    – speed up/down without pitch change
#   "lowpass"         – low-pass filter at cutoff Hz
#   "reverberation"   – simulated room reverb
#   "codec_mp3"       – MP3 codec compression
#   "codec_mulaw"     – mu-law 8-bit telephone codec

AUG_PARAMS = {
    "pitch_up":       {"semitones": 2},
    "pitch_down":     {"semitones": 2},
    "phase_noise":    {"noise_level": 0.012},
    "amp_scale":      {"scale": 1.20},
    "additive_noise": {"snr_db": 32},
    "time_stretch":   {"rate": 1.07},
    "lowpass":        {"cutoff_hz": 6500},
    "reverberation":  {"t60": 0.6, "room_scale": 0.5},
    "codec_mp3":      {"bitrate": "32k"},
    "codec_mulaw":    {},
}
# ─────────────────────────────────────────────────────────────────────────────

def aug_pitch_up(wav, semitones):
    factor = 2 ** (semitones / 12)
    frac   = Fraction(factor).limit_denominator(20)
    stretched = F.resample(wav, frac.numerator, frac.denominator)
    L = wav.shape[-1]
    if stretched.shape[-1] < L:
        stretched = torch.nn.functional.pad(stretched, (0, L - stretched.shape[-1]))
    return stretched[..., :L]

def aug_pitch_down(wav, semitones):
    factor = 2 ** (-semitones / 12)
    frac   = Fraction(factor).limit_denominator(20)
    stretched = F.resample(wav, frac.numerator, frac.denominator)
    L = wav.shape[-1]
    if stretched.shape[-1] < L:
        stretched = torch.nn.functional.pad(stretched, (0, L - stretched.shape[-1]))
    return stretched[..., :L]

def aug_phase_noise(wav, noise_level):
    spec  = torch.fft.rfft(wav)
    phase = torch.angle(spec)
    mag   = torch.abs(spec)
    phase = phase + torch.randn_like(phase) * noise_level * torch.pi
    return torch.fft.irfft(mag * torch.exp(1j * phase), n=wav.shape[-1])

def aug_amp_scale(wav, scale):
    return wav * scale

def aug_additive_noise(wav, snr_db):
    sig_power   = wav.pow(2).mean()
    noise       = torch.randn_like(wav)
    noise_power = noise.pow(2).mean()
    scale       = (sig_power / (noise_power * 10 ** (snr_db / 10))).sqrt()
    return wav + scale * noise

def aug_time_stretch(wav, rate):
    orig_len = wav.shape[-1]
    frac     = Fraction(rate).limit_denominator(20)
    stretched = F.resample(wav, frac.numerator, frac.denominator)
    if stretched.shape[-1] < orig_len:
        stretched = torch.nn.functional.pad(stretched, (0, orig_len - stretched.shape[-1]))
    return stretched[..., :orig_len]

def aug_lowpass(wav, cutoff_hz):
    return F.lowpass_biquad(wav, SAMPLE_RATE, cutoff_freq=cutoff_hz)

def aug_reverberation(wav, t60=0.6, room_scale=0.5):
    B, C, T = wav.shape
    rir_len = min(int(t60 * SAMPLE_RATE), 1600)
    t       = torch.linspace(0, t60, rir_len, device=wav.device)
    decay   = torch.exp(-6.9 * t / t60)
    rir     = torch.randn(rir_len, device=wav.device) * decay
    for delay_ms in [15, 30, 50]:
        d = int(delay_ms * 1e-3 * SAMPLE_RATE * room_scale)
        if d < rir_len:
            rir[d] += 0.4 * room_scale * decay[d]
    rir   = rir / (rir.abs().max() + 1e-8)
    n_fft = 2 ** math.ceil(math.log2(T + rir_len - 1))
    wav_f = torch.fft.rfft(wav, n=n_fft)
    rir_f = torch.fft.rfft(rir, n=n_fft)
    out   = torch.fft.irfft(wav_f * rir_f, n=n_fft)
    return out[..., :T]

def aug_codec_mp3(wav, bitrate="32k"):
    import io
    out_chunks = []
    for b in range(wav.shape[0]):
        buf = io.BytesIO()
        torchaudio.save(buf, wav[b].cpu(), SAMPLE_RATE, format="mp3",
                        compression=int(bitrate.replace("k", "")))
        buf.seek(0)
        decoded, sr = torchaudio.load(buf, format="mp3")
        if sr != SAMPLE_RATE:
            decoded = F.resample(decoded, sr, SAMPLE_RATE)
        L = wav.shape[-1]
        if decoded.shape[-1] < L:
            decoded = torch.nn.functional.pad(decoded, (0, L - decoded.shape[-1]))
        out_chunks.append(decoded[..., :L].unsqueeze(0))
    return torch.cat(out_chunks, dim=0).to(wav.device)

def aug_codec_mulaw(wav):
    wav8    = F.resample(wav, SAMPLE_RATE, 8000)
    encoded = torchaudio.functional.mu_law_encoding(wav8, quantization_channels=256)
    decoded = torchaudio.functional.mu_law_decoding(encoded, quantization_channels=256)
    wav16   = F.resample(decoded, 8000, SAMPLE_RATE)
    L = wav.shape[-1]
    if wav16.shape[-1] < L:
        wav16 = torch.nn.functional.pad(wav16, (0, L - wav16.shape[-1]))
    return wav16[..., :L]

# ─────────────────────────────────────────────────────────────────────────────

AUGMENTATIONS = {
    "pitch_up":       lambda w: aug_pitch_up(w,        **AUG_PARAMS["pitch_up"]),
    "pitch_down":     lambda w: aug_pitch_down(w,      **AUG_PARAMS["pitch_down"]),
    "phase_noise":    lambda w: aug_phase_noise(w,     **AUG_PARAMS["phase_noise"]),
    "amp_scale":      lambda w: aug_amp_scale(w,       **AUG_PARAMS["amp_scale"]),
    "additive_noise": lambda w: aug_additive_noise(w,  **AUG_PARAMS["additive_noise"]),
    "time_stretch":   lambda w: aug_time_stretch(w,    **AUG_PARAMS["time_stretch"]),
    "lowpass":        lambda w: aug_lowpass(w,         **AUG_PARAMS["lowpass"]),
    "reverberation":  lambda w: aug_reverberation(w,   **AUG_PARAMS["reverberation"]),
    "codec_mp3":      lambda w: aug_codec_mp3(w,       **AUG_PARAMS["codec_mp3"]),
    "codec_mulaw":    lambda w: aug_codec_mulaw(w),
}

assert ACTIVE_AUG in AUGMENTATIONS, \
    f"Unknown augmentation '{ACTIVE_AUG}'. Choose from: {list(AUGMENTATIONS.keys())}"

aug_fn        = AUGMENTATIONS[ACTIVE_AUG]
active_params = AUG_PARAMS[ACTIVE_AUG]

# ── Run selected augmentation over full val dataloader ───────────────────────
chunks = []
print(f"Running augmentation: '{ACTIVE_AUG}' | params: {active_params}\n")

for batch_idx, batch in enumerate(val_dataloader):
    wav = batch["audio"]
    if wav.dim() == 2:
        wav = wav.unsqueeze(1)
    try:
        chunks.append(aug_fn(wav.clone()).cpu())
    except Exception as e:
        print(f"  [WARN] batch {batch_idx} failed: {e}")

    if (batch_idx + 1) % 10 == 0:
        print(f"  processed ~{(batch_idx + 1) * val_dataloader.batch_size} samples …")

# Concat
if not chunks:
    raise ValueError("No augmented batches were produced; check warnings above.")

max_len = max(c.shape[-1] for c in chunks)
aug_tensor = torch.cat(
    [torch.nn.functional.pad(c, (0, max_len - c.shape[-1])) for c in chunks], dim=0
)
print(f"\n✅  '{ACTIVE_AUG}' {active_params}  →  tensor shape: {aug_tensor.shape}")

# ── Audition 10 random clips ─────────────────────────────────────────────────
listen_idx = random.sample(range(aug_tensor.shape[0]), min(LISTEN_N, aug_tensor.shape[0]))
print(f"\n🎧  {len(listen_idx)} random clips  ──  {ACTIVE_AUG} {active_params}\n{'─'*50}")

for rank, idx in enumerate(listen_idx, 1):
    wav_np = aug_tensor[idx, 0].numpy().astype(np.float32)
    peak   = np.abs(wav_np).max()
    if peak > 0:
        wav_np /= peak
    print(f"  clip {rank:>2d}  (val idx {idx})")
    display(ipd.Audio(wav_np, rate=SAMPLE_RATE, normalize=False))

Running augmentation: 'pitch_down' | params: {'semitones': 2}

  processed ~200 samples …
  processed ~400 samples …
  processed ~600 samples …
  processed ~800 samples …
  processed ~1000 samples …
  processed ~1200 samples …
  processed ~1400 samples …
  processed ~1600 samples …
  processed ~1800 samples …
  processed ~2000 samples …
  processed ~2200 samples …
  processed ~2400 samples …
  processed ~2600 samples …
  processed ~2800 samples …
  processed ~3000 samples …

✅  'pitch_down' {'semitones': 2}  →  tensor shape: torch.Size([3000, 1, 48000])

🎧  10 random clips  ──  pitch_down {'semitones': 2}
──────────────────────────────────────────────────
  clip  1  (val idx 2619)


  clip  2  (val idx 456)


  clip  3  (val idx 102)


  clip  4  (val idx 1126)


  clip  5  (val idx 1003)


  clip  6  (val idx 914)


  clip  7  (val idx 571)


  clip  8  (val idx 419)


  clip  9  (val idx 2771)


  clip 10  (val idx 2233)


# Lit model

The settings of the `AudioModel` are defined in the `cfg`. Each setting is a key-value pair, where the key is the name of the setting and the value is the value of the setting. The meaning of each setting is defined as follows:

1. **Network Structure Parameters**:
   - `backbone` : "wavlm", the backbone of the phoneme recognition model.
   - `use_raw` : `False`, whether to use raw transformer as the backbone
   - `use_GAT`: `True`, whether to use GAT
   - `n_edges`: `10`, the nubmer of edges for each node in the GAT
   - `use_pool`: `True`, whether to use pooling


2. **Loss Function Parameters**:
   - `use_clip`: `True`, whether to use clip loss


3. **Data Augmentation and Training Strategy**:
   - `use_aug`: `True`, whether to use data augmentation in the training


In [36]:
from argparse import Namespace

# Construct the configuration using Namespace
cfg = Namespace(
    PhonemeGAT=Namespace(
        backbone="wavlm",  # wavlm or wav2vec
        use_raw=False,              # whether to use raw transformer as the backbone
        use_GAT=True,              # whether to use GAT
        n_edges=10,                # the nubmer of edges for each node in the GAT
        use_aug=True,              # whether to use data augmentation in the training
        use_pool=True,            # whether to use pooling
        use_clip=True,             # whether to use clip loss
    )
)

In [37]:
from pytorch_lightning import Trainer, LightningModule
from pytorch_lightning.loggers import  CSVLogger

We use the pytorch Lightning module to train the model, where we define the train step, validation/predict step, loss function and optimizer.

In [38]:
audio_model_lit = Phoneme_GAT_lit(cfg=cfg)

Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['lm_head.bias', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.weight', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


## Test forwarding 

In the lit model, we use the `_shared_pred` method to predict the logits of the input batch. If the stage is train, we also the the audio_transform to augment the spectrogram.

Generate a random batch:

In [39]:
x = torch.randn(3, 1, 48000)
batch = {
    "label": torch.randint(0, 2, (3,)),
    "audio": x,
    "sample_rate": 16000,
}

Note, you batch must be a dict with above keys.

In [40]:
batch_res = audio_model_lit._shared_pred(batch=batch, batch_idx=0)
for key, value in batch_res.items():
    print(key, value.shape)

logit torch.Size([3])
hidden_states torch.Size([3, 768])
phoneme_feat torch.Size([3, 149, 768])
encoder_feat torch.Size([3, 149, 768])
phoneme_cls_logit torch.Size([3, 687])
phoneme_cls_label torch.Size([3])
aug_logit torch.Size([3])
aug_frame_logit torch.Size([3])
aug_labels torch.Size([3])


## Demo training

We first build a simple dataloaders for training, where all the samples are randomly generated.

In [41]:
# from callbacks import EER_Callback, BinaryAUC_Callback, BinaryACC_Callback

# There code was balls so I rewrote it to be simple
from callbacks_rational import BinaryACC_Callback, BinaryAUC_Callback, EER_Callback, TPR_Callback, TNR_Callback, FPR_Callback, FNR_Callback

In [42]:
import torch
from torch.utils.data import Dataset, DataLoader

In [43]:
class SimpleTestDataset(Dataset):
    def __init__(self, num_samples=10):
        # Generate synthetic data similar to your example
        self.samples = []
        for _ in range(num_samples):
            self.samples.append({
                "audio": torch.randn(1, 48000),
                "label": torch.randint(0, 2, (1,)).item(),
                "sample_rate": 16000,
            })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]



# Create the dataset and dataloader
test_dataset = SimpleTestDataset(num_samples=20)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=3,
    shuffle=False,
)

We build a simple trainer to train and test our model.

In [80]:
import wandb
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Finish any existing wandb run
if wandb.run is not None:
    wandb.finish()

wandb_logger = WandbLogger(
    project="DeepfakeDetectionRenewed",
    entity="krishrawat0222-f",
    name=f"attack_{ACTIVE_AUG}",
    log_model=False,
    tags=["attack", "validation", ACTIVE_AUG],
)

trainer = Trainer(
    logger=wandb_logger,
    callbacks=[
        BinaryACC_Callback(batch_key="label", output_key="logit"),
        BinaryAUC_Callback(batch_key="label", output_key="logit"),
        EER_Callback(batch_key="label", output_key="logit"),
        TPR_Callback(batch_key="label", output_key="logit"),
        TNR_Callback(batch_key="label", output_key="logit"),
        FPR_Callback(batch_key="label", output_key="logit"),
        FNR_Callback(batch_key="label", output_key="logit"),
    ],
)

epoch,▁
trainer/global_step,▁
val-acc,▁
val-auc,▁
val-aug_loss,▁
val-clip_loss,▁
val-cls_loss,▁
val-eer,▁
val-fnr,▁
val-fpr,▁
+3,...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [81]:
CKPT_PATH = "goat.ckpt"

model = Phoneme_GAT_lit.load_from_checkpoint(CKPT_PATH)
model = model.cuda()
torch.set_float32_matmul_precision('medium')
model.eval()
from torch.utils.data import DataLoader

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, aug_fn):
        self.base    = base_dataset
        self.aug_fn  = aug_fn

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        item = self.base[idx]
        wav  = item["audio"]                        # (1, T)
        wav  = self.aug_fn(wav.unsqueeze(0))        # add batch dim → (1,1,T)
        item["audio"] = wav.squeeze(0)              # back to (1, T)
        return item

# Wrap the existing val dataset
aug_val_dataset    = AugmentedDataset(val_dataloader.dataset, aug_fn)
aug_val_dataloader = DataLoader(
    aug_val_dataset,
    batch_size=val_dataloader.batch_size,
    shuffle=False,
    num_workers=val_dataloader.num_workers,
)

Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['lm_head.bias', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.weight', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


In [82]:
print(next(model.parameters()).device)

cuda:0


In [83]:


# Now validate on augmented inputs
trainer.validate(model=model, dataloaders=aug_val_dataloader)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=29` in the `DataLoader` to improve performance.


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val-acc            0.9359999895095825
         val-auc            0.9829244613647461
      val-aug_loss                  0.0
      val-clip_loss        0.047543056309223175
      val-cls_loss          0.17782479524612427
         val-eer            0.06533333659172058
         val-fnr            0.09066666662693024
         val-fpr            0.03733333200216293
        val-loss            0.20159633457660675
         val-tnr                    0.0
         val-tpr                    1.0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'val-loss': 0.20159633457660675,
  'val-cls_loss': 0.17782479524612427,
  'val-clip_loss': 0.047543056309223175,
  'val-aug_loss': 0.0,
  'val-acc': 0.9359999895095825,
  'val-auc': 0.9829244613647461,
  'val-eer': 0.06533333659172058,
  'val-tpr': 1.0,
  'val-tnr': 0.0,
  'val-fpr': 0.03733333200216293,
  'val-fnr': 0.09066666662693024}]

After training, you can view the logging loss in the logger file, for example `logs/lightning_logs/version_0/metrics.csv`.
![](imgs/loss.png)

In [ ]:
import wandb
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger
from torch.utils.data import DataLoader

model = model.cuda()
torch.set_float32_matmul_precision('medium')

SKIP_AUGS = {"pitch_up", "pitch_down", "codec_encodec"}

for aug_name, aug_fn_i in AUGMENTATIONS.items():
    if aug_name in SKIP_AUGS:
        print(f"  [SKIP] {aug_name}")
        continue

    print(f"\n{'='*60}")
    print(f"Running attack: {aug_name} | params: {AUG_PARAMS.get(aug_name, {})}")
    print(f"{'='*60}")

    aug_val_dataset    = AugmentedDataset(val_dataloader.dataset, aug_fn_i)
    aug_val_dataloader = DataLoader(
        aug_val_dataset,
        batch_size=val_dataloader.batch_size,
        shuffle=False,
        num_workers=0,
    )

    if wandb.run is not None:
        wandb.finish()

    wandb_logger = WandbLogger(
        project="DeepfakeDetectionRenewed",
        entity="krishrawat0222-f",
        name=f"attack_{aug_name}",
        log_model=False,
        tags=["attack", "validation", aug_name],
    )

    trainer = Trainer(
        accelerator="gpu",
        devices=1,
        logger=wandb_logger,
        callbacks=[
            BinaryACC_Callback(batch_key="label", output_key="logit"),
            BinaryAUC_Callback(batch_key="label", output_key="logit"),
            EER_Callback(batch_key="label", output_key="logit"),
            TPR_Callback(batch_key="label", output_key="logit"),
            TNR_Callback(batch_key="label", output_key="logit"),
            FPR_Callback(batch_key="label", output_key="logit"),
            FNR_Callback(batch_key="label", output_key="logit"),
        ],
    )

    try:
        trainer.validate(model=model, dataloaders=aug_val_dataloader)
    except Exception as e:
        print(f"  [WARN] {aug_name} failed: {e}")
        continue

if wandb.run is not None:
    wandb.finish()

print("\n✅  All augmentation attacks complete. Check WandB dashboard.")